# DICOM Tag Addition Notebook

Customer/operator walkthrough for adding DICOM tags end to end. This example adds Requested Procedure Description (0032,1060), Procedure Code Sequence (0008,1032), and Requested Procedure Code Sequence (0032,1064).

This notebook is instructional and uses SQL/Spark examples. Review each step, validate PHI/governance, and execute only in an approved environment.

## Workflow

1. Inspect source population.
2. Insert tag dictionary rows in Admin.
3. Add governance metadata.
4. Add extraction configuration/schema.
5. Validate structured sequence handling.
6. Deploy the extraction notebook.
7. Run a bounded study test.
8. Verify tag coverage and source/target counts.
9. Materialize reporting tables.
10. Add the fields to the reporting/search layer.

Do not skip the bounded test. Do not run a full rebuild until the test passes.

## 1. Source reconnaissance

Use Spark/Delta for authoritative metadata inspection. SQL endpoint JSON parsing is not the extraction path because metadata payloads can be truncated or invalid in SQL projections.

Check the three tags in Bronze/Silver and record population, value shape, and representative Study UIDs.

In [ ]:
# Fabric/Spark example
from pyspark.sql import functions as F

for tag in ["00321060", "00081032", "00321064"]:
    print(tag)
    # Inspect metadata JSON with Spark; select a bounded sample for review.
    # source.where(F.col("metadata_string").contains(tag)).select("studyInstanceUid", "metadata_string").limit(5).show(truncate=False)


## 2. Insert the Admin dictionary rows

The dictionary controls enabled extraction. The sequence tags are stored as JSON strings in the wide target so multiple sequence items are not lost.

Run these statements through an approved write-capable Spark/Delta process, not a read-only SQL analytics endpoint.

In [ ]:
-- SQL reference statements; execute through an approved Admin Delta/Spark write path.
INSERT INTO healthcare1_msft_admin.dbo.DicomTagDictionary
(dicomTag, dicomKeyword, canonicalColumnName, vrExpected, scope, isWellKnown29, isPhi, phiCategory, enabled, createdAt, updatedAt)
VALUES
('00321060','RequestedProcedureDescription','requestedProcedureDescription','LO','procedure',0,0,'None',1,SYSUTCDATETIME(),SYSUTCDATETIME()),
('00081032','ProcedureCodeSequence','procedureCodeSequence','SQ','procedure',0,0,'None',1,SYSUTCDATETIME(),SYSUTCDATETIME()),
('00321064','RequestedProcedureCodeSequence','requestedProcedureCodeSequence','SQ','procedure',0,0,'None',1,SYSUTCDATETIME(),SYSUTCDATETIME());

-- Confirm no duplicate enabled tags before continuing.
SELECT dicomTag, COUNT_BIG(*) AS duplicateRows
FROM healthcare1_msft_admin.dbo.DicomTagDictionary
WHERE enabled = 1
GROUP BY dicomTag
HAVING COUNT_BIG(*) > 1;

## 3. Governance and sensitivity

Add governance rows for the three target columns. Requested Procedure Description is readable clinical text; confirm customer policy before classifying it as General. The coded sequences are not normally direct identifiers, but retain customer review responsibility.

In [ ]:
INSERT INTO healthcare1_msft_admin.dbo.ImagingMetastoreExtensionGovernance
(targetTable, columnName, dicomTag, dicomKeyword, isPhi, phiCategory, recommendedSensitivityLabel, createdAt)
VALUES
('ImagingMetastoreExtension','requestedProcedureDescription','00321060','RequestedProcedureDescription',0,'None','General',SYSUTCDATETIME()),
('ImagingMetastoreExtension','procedureCodeSequence','00081032','ProcedureCodeSequence',0,'None','General',SYSUTCDATETIME()),
('ImagingMetastoreExtension','requestedProcedureCodeSequence','00321064','RequestedProcedureCodeSequence',0,'None','General',SYSUTCDATETIME());

## 4. Update extraction configuration and target schema

Add the three entries to the extraction notebook tag configuration. Add target columns:

- requestedProcedureDescription: string
- procedureCodeSequence: string containing JSON
- requestedProcedureCodeSequence: string containing JSON

The implementation must use the deterministic composite sourceRecordKey (source system + Study UID + Series UID + SOP UID + file path). Do not restore the old ImagingMetastore.id-only merge key.

In [ ]:
# Expected configuration entries in SEED_TAGS
NEW_TAGS = [
    ("00321060", "RequestedProcedureDescription", "requestedProcedureDescription", "LO", "procedure", False, False, "None", True),
    ("00081032", "ProcedureCodeSequence", "procedureCodeSequence", "SQ", "procedure", False, False, "None", True),
    ("00321064", "RequestedProcedureCodeSequence", "requestedProcedureCodeSequence", "SQ", "procedure", False, False, "None", True),
]

# For SQ tags, preserve the complete Value object as JSON.
# Do not select only Value[0] unless the business requirement explicitly accepts data loss.

## 5. Sequence parsing contract

RequestedProcedureDescription (0032,1060) is usually a scalar LO value. ProcedureCodeSequence (0008,1032) and RequestedProcedureCodeSequence (0032,1064) are SQ values and can contain multiple coded items.

Store the sequence columns as JSON strings first. A later reporting layer may flatten the first or all code items into code value, coding scheme, and code meaning fields.

In [ ]:
# Illustrative Spark expressions
from pyspark.sql import functions as F

description = F.get_json_object(F.col("metadata_json"), "$.00321060.Value[0]")
procedure_sequence_json = F.get_json_object(F.col("metadata_json"), "$.00081032.Value")
requested_sequence_json = F.get_json_object(F.col("metadata_json"), "$.00321064.Value")

# Use these expressions as the target columns. Validate that sequence JSON is non-null and valid.

## 6. Deploy the updated extraction notebook

Deploy the source after code/schema review. The deployment keeps the Fabric item ID stable.

In [ ]:
# PowerShell
./deploy-imaging-metastore-extension-notebook.ps1 `
  -FabricWorkspaceName "FUJIV_Fabric_Test" `
  -SilverLakehouseName "healthcare1_msft_silver" `
  -AdminLakehouseName "healthcare1_msft_admin" `
  -NotebookName "02_extract_imaging_metastore_extension" `
  -NotebookFolderName "DICOM Tag Extension"

## 7. Run a bounded validation

Use 3–10 representative Study Instance UIDs. Include one study with a description, one with coded sequence content, one with multiple sequence items, and one with no new tags.

Pass FILTER_STUDY_INSTANCE_UIDS and keep DRY_RUN_ONLY=false to validate the target write. Filtered runs do not advance the production watermark.

In [ ]:
# PowerShell example
./deploy-imaging-metastore-extension-notebook.ps1 `
  -FabricWorkspaceName "FUJIV_Fabric_Test" `
  -SilverLakehouseName "healthcare1_msft_silver" `
  -AdminLakehouseName "healthcare1_msft_admin" `
  -NotebookName "02_extract_imaging_metastore_extension" `
  -RunAfterDeploy `
  -NotebookParameters @{ `
    FILTER_STUDY_INSTANCE_UIDS = "uid1,uid2,uid3"; `
    MAX_SOURCE_ROWS_PER_BATCH = "50000"; `
    DRY_RUN_ONLY = "false" `
  }

## 8. Verify the new values

Check that the requested description is readable and the sequence columns contain valid JSON. Confirm existing tag coverage, UID validation, ImagingStudy joins, and sourceRecordKey uniqueness.

In [ ]:
SELECT TOP (100)
    studyInstanceUid, seriesInstanceUid, sopInstanceUid,
    requestedProcedureDescription,
    procedureCodeSequence,
    requestedProcedureCodeSequence,
    extractRunId, extractedAt
FROM healthcare1_msft_silver.dbo.ImagingMetastoreExtension
WHERE studyInstanceUid IN ('uid1','uid2','uid3')
ORDER BY extractedAt DESC;

SELECT COUNT_BIG(*) AS duplicateSourceKeys
FROM (SELECT sourceRecordKey FROM healthcare1_msft_silver.dbo.ImagingMetastoreExtension WHERE isActive=1 GROUP BY sourceRecordKey HAVING COUNT_BIG(*) > 1) d;

## 9. Materialize reporting and update the report

Run the materializer after validation. It refreshes inventory/search/log tables and tag metrics in Admin.

Then refresh Admin SQL metadata and deploy/refresh the Power BI semantic model. Add the three fields to the Ingestion Analytics & Search page:

- Requested Procedure Description
- Procedure Code Sequence
- Requested Procedure Code Sequence

Recommended report treatment: show Requested Procedure Description directly; expose coded sequence fields in a detail/drill-through view rather than as primary KPI cards.

In [ ]:
# PowerShell
./Deploy-DicomTagExtensionNotebook.ps1 `
  -FabricWorkspaceName "FUJIV_Fabric_Test" `
  -SilverLakehouseName "healthcare1_msft_silver" `
  -AdminLakehouseName "healthcare1_msft_admin" `
  -NotebookName "05_materialize_imaging_ingestion_report" `
  -SourceFile "materialize_imaging_ingestion_report.py" `
  -RunAfterDeploy

## 10. Production acceptance checklist

- Dictionary rows exist once and are enabled.
- Governance rows exist and classification is approved.
- Target columns exist.
- Description values are readable.
- SQ sequence columns preserve valid JSON and multiple items.
- Bounded run has zero join, UID, and parse failures.
- Composite sourceRecordKey has no duplicates.
- Filtered validation did not advance the watermark.
- Tag coverage metrics are populated.
- Report search/detail view displays the new fields.
- Full production run is approved only after bounded validation passes.